# Notebook data_cleaning
This notebook cleans and prepares the raw Wildfire data and the Provinces and Territories of Canada data for further analysis.

### Input data
- NFDB_point.csv: Raw wildfire data of Canada
- lpr_000b21a_e.zip: Raw province and territory data of Canada

### Outputs
- canada_4326.gpkg: Provinces and Territories of Canada data with crs EPSG:4326
- fire_clean.csv: Cleaned wildfire data
- fire_14_23.csv: Cleaned wildfire data for the years 2014 - 2023

### Key assumptions
- NFDB_point.csv (province/territory data of Canada) is reprojected to EPSG:4326

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# 1. Loading data
# Load wildfire data
wildfire = pd.read_csv("../data/raw/NFDB_point.csv", sep = ";")
print("Data loaded successfully")

# Load Canada data
canada = gpd.read_file("../data/raw/lpr_000b21a_e.zip")
print("Zip loaded successfully")

Data loaded successfully
Zip loaded successfully


In [3]:
# 2. Reproject canada data to epsg:4326 to ensure spatial compatibility between the datasets and enable interactive web mapping with Folium
Source_CRS = "EPSG:3347"
TARGET_CRS = "EPSG:4326"

canada = canada.to_crs(TARGET_CRS)

# Check if the reprojection worked
print(canada.crs)

EPSG:4326


In [4]:
# 3. Exporting Canada data with epsg 4326
canada.to_file("../data/processed/canada_4326.gpkg", driver = "GPKG")
print("Export canada_4326 successful")

Export canada_4326 successful


## Data cleaning
The following sections 4.1 - 4.6 are cleaning the Wildfire data step by step.

Section 4.1:
- Dropping colums which are not needed in this project
- Renaiming the remaind colums

Section 4.2:
- Cleaning the column of 'cause' --> Dropping the rows with H-PB (= Prescribed Burn) as cause, as they are fires explicitly made by humans and are not of interesst for this project
- Dropping the row which has a wrong latitude (latitude is not in Canada)

Section 4.3:
- Checking if the data has NaNs

Section 4.4:
- Turning the initials of the column 'provinces' into full names

Section 4.5:
- In the column of 'province' "Parc Canada" is not a province, but wildfires wich happened in National Parcs of Canada --> Thoes Wildfires have to be assigned to their correct province

Section 4.6:
- Turning the initials of the column 'cause' into full names

In [5]:
# 4.1 Data cleaning wildfire data part 1
# Dropping columns which are not needed for this project
wildfire_clean = wildfire.drop(["FID",
                        "the_geom",
                        "NFDBFIREID",
                        "NAT_PARK",
                        "FIRENAME",
                        "MONTH",
                        "DAY",
                        "REP_DATE",
                        "OUT_DATE",
                        "FIRE_TYPE",
                        "RESPONSE",
                        "PROTZONE",
                        "MORE_INFO"],
axis = 1) # axis = 1 --> columns


# Renaming columns
new_names = {
    "SRC_AGENCY": "province",
    "FIRE_ID": "fire_id",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "YEAR": "year",
    "CAUSE": "cause",
    "SIZE_HA": "size_ha",
}

wildfire_clean = wildfire_clean.rename(columns = new_names)

# Check if the renaming worked
wildfire_clean.columns

Index(['province', 'fire_id', 'latitude', 'longitude', 'year', 'size_ha',
       'cause'],
      dtype='str')

In [6]:
# 4.2 Data cleaning wildfire data part 2
# Dropping rows which have H-PB (= Prescribed Burn) as cause
wildfire_clean = wildfire_clean[wildfire_clean['cause'] != 'H-PB']
print((wildfire_clean['cause'] == 'H-PB').sum())

# Dropping row with has a wrong longitude
fire_clean = wildfire_clean.drop(wildfire_clean[wildfire_clean["longitude"] > 0].index)

# Check if the dropping of the wrong longitude worked
print(f"Rows in wildfire_clean:{len(wildfire_clean)}")
print(f"Rows in fire_clean:{len(fire_clean)}")

0
Rows in wildfire_clean:15138
Rows in fire_clean:15137


In [7]:
# 4.3 Data cleaning wildfire data part 3
# Checking if a column has NaNs
print(fire_clean["province"].hasnans)
print(fire_clean["fire_id"].hasnans)
print(fire_clean["latitude"].hasnans)
print(fire_clean["longitude"].hasnans)
print(fire_clean["year"].hasnans)
print(fire_clean["size_ha"].hasnans)
print(fire_clean["cause"].hasnans)

False
False
False
False
False
False
False


In [8]:
# 4.4 Data cleaning wildfire data part 4
# Turning the initials of provinces into full names
province_full = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "ON": "Ontario",
    "PC": "Parc Canada",
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon"
}

fire_clean["province"] = fire_clean["province"].map(province_full).fillna(fire_clean["province"])

# Check if the renaming worked and how many wildfires are under "Parc Canada"
print(fire_clean["province"].value_counts())

province
Northwest Territories        2524
Saskatchewan                 2392
Manitoba                     2222
Ontario                      1816
British Columbia             1755
Quebec                       1485
Yukon                        1252
Alberta                       930
Parc Canada                   433
Newfoundland and Labrador     271
New Brunswick                  39
Nova Scotia                    18
Name: count, dtype: int64


In [9]:
# 4.5. Data cleaning wildfire data part 5
# Assigning "Parc Canada" wildfires to the correct provinces
mask = fire_clean["province"] == "Parc Canada" # Only choose rows with "Parc Canada"

# Making GeoDataFrame
geometry = [Point(xy) for xy in zip
            (fire_clean.loc[mask, "longitude"],
             fire_clean.loc[mask, "latitude"])
]

gdf_points = gpd.GeoDataFrame(
    fire_clean.loc[mask].copy(),
    geometry=geometry,
    crs="EPSG:4326"
)

# IMPORTANT: Check column names of the province/territory canada data to see which column includes the provinces
print(canada.columns)

# Spatial join
joined = gpd.sjoin(
    gdf_points,
    canada,
    how="left",
    predicate="within"
)

fire_clean.loc[mask, "province"] = joined["PRENAME"].values

# Check if there are no wildfires for "Parc Canada" --> if there are non, the new assignement worked
print(fire_clean["province"].value_counts())
print((fire_clean["province"] == "Parc Canada").sum())

Index(['PRUID', 'DGUID', 'PRNAME', 'PRENAME', 'PRFNAME', 'PREABBR', 'PRFABBR',
       'LANDAREA', 'geometry'],
      dtype='str')
province
Northwest Territories        2640
Saskatchewan                 2414
Manitoba                     2238
Ontario                      1817
British Columbia             1780
Quebec                       1486
Yukon                        1256
Alberta                      1176
Newfoundland and Labrador     271
New Brunswick                  39
Nova Scotia                    19
Name: count, dtype: int64
0


In [10]:
# 4.6 Data cleaning wildfire data part 6
# Turning the initials of cause into full names
cause_full = {
    "H": "Human",
    "N": "Natural",
    "U": "Unknown"
}

fire_clean["cause"] = fire_clean["cause"].map(cause_full).fillna(fire_clean["cause"])

In [11]:
# 5. Exporting cleand fire data
fire_clean.to_csv("../data/processed/fire_clean.csv", index = False)
print("Export successful")

Export successful


In [12]:
# 6. For further analysis only the years 2014 - 2023 are of interest, so I only keep thoes years
# To know which years are available
print(fire_clean["year"].unique())

# Dropping row with years I don't need
fire_14_23 = fire_clean.drop(fire_clean[fire_clean["year"] <2014].index)

# To know if the dropping worked
print(fire_14_23["year"].unique())

[2023 2022 2021 2020 2019 2018 2017 2016 2015 2014 2013 2012 2011 2010
 2009 2008 2007 2006 2005 2004 2003 2002 2001 2000 1999 1998 1997 1996
 1995 1994 1993 1992 1991 1990 1989 1988 1987 1986 1985 1984 1983 1982
 1981 1980]
[2023 2022 2021 2020 2019 2018 2017 2016 2015 2014]


In [13]:
# 7. Exporting fire data for 2014-2023
fire_14_23.to_csv("../data/processed/fire_14_23.csv", index = False)
print("Export successful")

Export successful
